In [ ]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import fiona
import pyproj
from osgeo import gdal
from area import area

In [ ]:
def shp_to_gdf(shapefile_path, driver):
    shapefile_features =[]
    with fiona.collection(shapefile_path, driver=driver) as shp_in:
        shp_crs = pyproj.crs.CRS(shp_in.crs)
        shpschema = shp_in.schema.copy()
        for obj in shp_in:
            shapefile_features.append(obj)
    shp_gdf = gpd.GeoDataFrame.from_features([feature for feature in shapefile_features], crs=shp_crs)
    shp_gdf['bounds'] = shp_gdf.geometry.apply(lambda x: x.bounds)
    shp_gdf['bounds'] = [list(i) for i in shp_gdf['bounds']]
    return shp_gdf

def get_intersection_area(poly_1, poly_2):
    if not poly_1.is_valid:
        poly_1 = poly_1.buffer(0)
    if not poly_2.is_valid:
        poly_2 = poly_2.buffer(0)
    return (poly_1.intersection(poly_2).area)/10**4

In [ ]:
data_path = "../data/all_projects_TM_20240614/all_projects_TM.shp"

In [ ]:
shapefile_features =[]
with fiona.collection(data_path, 'r') as shp_in:
    shp_crs = pyproj.crs.CRS(shp_in.crs)
    shpschema = shp_in.schema.copy()
    for obj in shp_in:
        shapefile_features.append(obj)
shp_gdf = gpd.GeoDataFrame.from_features([feature for feature in shapefile_features], crs=shp_crs)
#shp_gdf['bounds'] = shp_gdf.geometry.apply(lambda x: x.bounds)
#shp_gdf['bounds'] = [list(i) for i in shp_gdf['bounds']]
shp_gdf['plantstart'] = pd.to_datetime(shp_gdf['plantstart'], format='mixed').dt.date

In [ ]:
project_list = shp_gdf['Project'].unique()

In [ ]:
df_short_list = []
for project in project_list:
    project_gdf = shp_gdf[shp_gdf['Project'] == project]
    project_gdf['Project'] = project
    project_gdf_copy = project_gdf.copy()
    project_gdf_copy['geometry'] = project_gdf_copy['geometry'].to_crs({'init': 'epsg:3857'})
    project_gdf['Area_ha'] = project_gdf_copy['geometry'].apply(lambda x: x.area/10**4)
    df_short_list.append(project_gdf)

all_proj_df = gpd.GeoDataFrame(pd.concat(df_short_list, ignore_index=True))
keep_columns = ['geometry', 'Project', 'SiteName', 'poly_name', 'plantstart',
        'target_sys', 'practice', 'Area_ha']
#long_colnames_keep = [item for item in all_proj_df.columns if item in keep_columns]
all_proj_df= all_proj_df[keep_columns]
all_proj_df = all_proj_df[~pd.isna(all_proj_df['poly_name'])]

all_proj_df['practice'] = all_proj_df['practice'].apply(lambda x: x.replace(' ', '').split(","))
all_proj_df['target_sys'] = all_proj_df['target_sys'].apply(lambda x: x.replace(' ', '').split(","))

In [ ]:
practice = all_proj_df['practice'].apply(pd.Series).reset_index().melt(id_vars='index').dropna()[['index', 'value']].set_index('index')
practice_gdf = all_proj_df.__deepcopy__().drop('practice', axis=1)
split_practice_project_gdf = pd.merge(practice_gdf, practice, left_index=True, right_index=True).rename(columns={'value': 'practice'})
colnames = split_practice_project_gdf.columns
remove = {'practice','Area_ha', 'bounds', 'target_sys'}
new_colnames = [item for item in colnames if item not in remove ]
practice_gdf_long = pd.pivot(split_practice_project_gdf, index = new_colnames, columns='practice', values='Area_ha').reset_index()
practice_gdf_long.index.name=None
practice_gdf_long.rename(columns={'Null':'practice_NA'}, inplace=True)

In [ ]:
target_sys = all_proj_df['target_sys'].apply(pd.Series).reset_index().melt(id_vars='index').dropna()[['index', 'value']].set_index('index')
target_sys_gdf = all_proj_df.__deepcopy__().drop('target_sys', axis=1)
split_target_sys_project_gdf = pd.merge(target_sys_gdf, target_sys, left_index=True, right_index=True).rename(columns={'value': 'target_sys'})
colnames = split_target_sys_project_gdf.columns
remove = {'practice','Area_ha', 'bounds', 'target_sys'}
new_colnames = [item for item in colnames if item not in remove ]
target_sys_gdf_long = pd.pivot(split_target_sys_project_gdf, index = new_colnames, columns='target_sys', values='Area_ha').reset_index()
target_sys_gdf_long.index.name=None
target_sys_gdf_long.rename(columns={'Null':'target_sys_NA'}, inplace=True)

In [ ]:
target_sys_gdf_long.columns

In [ ]:
practice_target_sys_poly = practice_gdf_long.merge(target_sys_gdf_long, on=['geometry', 'Project', 'SiteName', 'poly_name', 'plantstart'], how='left')
practice_target_sys_poly.columns
zero_fill_cols = ['practice_NA', 'assisted-natural-regeneration', 'direct-seeding','tree-planting', 'target_sys_NA', 'agroforest', 'natural-forest','riparian-area-or-wetland', 'woodlot-or-plantation']
practice_target_sys_poly[zero_fill_cols] = practice_target_sys_poly[zero_fill_cols].replace({np.nan:int(0)})

In [ ]:
practice_target_sys_by_project = practice_target_sys_poly.groupby('Project')[zero_fill_cols].sum().reset_index()
practice_target_sys_by_project

# Imagery availablity 

In [ ]:
imagery_data_path = "../data/afr100_cohort1_imagery_availability/"
imagery_files = os.listdir(imagery_data_path)
all_project_imagery = []
for project in imagery_files:
    project_df = pd.read_csv(f"{imagery_data_path}/{project}")
    project_sub_df = project_df[['Name', 'properties.datetime','collection', 'properties.eo:cloud_cover']]
    project_sub_df['Project'] = project.replace('afr100_', '').replace('_imagery_availability.csv', '')
    all_project_imagery.append(project_sub_df)
all_projects_imagery_df = pd.concat(all_project_imagery).reset_index()
all_projects_imagery_df = all_projects_imagery_df[['Project', 'Name', 'properties.datetime','collection', 'properties.eo:cloud_cover']]
all_projects_imagery_df = all_projects_imagery_df[~pd.isna(all_projects_imagery_df['Name'])]

all_projects_imagery_df.rename(columns={'Name':'poly_name'}, inplace=True)
all_projects_imagery_df['properties.datetime'] =pd.to_datetime(all_projects_imagery_df['properties.datetime'], format='mixed').dt.normalize()
all_projects_imagery_df['properties.datetime'] =all_projects_imagery_df['properties.datetime'].apply(lambda x: x.replace(tzinfo=None))

proj_plant_date = all_proj_df[['Project',  'poly_name', 'plantstart']]
all_proj_df_imagery = proj_plant_date.merge(all_projects_imagery_df, on=['Project', 'poly_name'], how='left')
all_proj_df_imagery = all_proj_df_imagery[~pd.isna(all_proj_df_imagery['plantstart'])]
all_proj_df_imagery = all_proj_df_imagery[all_proj_df_imagery['plantstart'] != '15-04-2024 - 15-05-2024']
all_proj_df_imagery['plantstart'] =pd.to_datetime(all_proj_df_imagery['plantstart'], format='mixed')
all_proj_df_imagery['properties.datetime'] =pd.to_datetime(all_proj_df_imagery['properties.datetime'], format='mixed').dt.normalize()
all_proj_df_imagery['properties.datetime'] =all_proj_df_imagery['properties.datetime'].apply(lambda x: x.replace(tzinfo=None))
all_proj_df_imagery['dateDiff']=(all_proj_df_imagery['properties.datetime'] - all_proj_df_imagery['plantstart']).dt.days

all_proj_df_imagery_usable_baseline = all_proj_df_imagery[(all_proj_df_imagery['dateDiff'] <0 ) & (all_proj_df_imagery['dateDiff'] > -365 ) & (all_proj_df_imagery['properties.eo:cloud_cover'] < 50)]
all_proj_df_imagery_usable_baseline_summary = all_proj_df_imagery_usable_baseline.groupby(['Project', 'poly_name']).count().reset_index()[['Project', 'poly_name', 'collection']]
all_proj_df_imagery_usable_baseline_summary.rename(columns={'collection':'available_baseline_images'}, inplace=True)

all_proj_df_imagery_usable_ev = all_proj_df_imagery[(all_proj_df_imagery['dateDiff'] > 90 ) & (all_proj_df_imagery['properties.eo:cloud_cover'] < 50)]
all_proj_df_imagery_usable_ev_summary = all_proj_df_imagery_usable_ev.groupby(['Project', 'poly_name']).count().reset_index()[['Project', 'poly_name', 'collection']]
all_proj_df_imagery_usable_ev_summary.rename(columns={'collection':'available_ev_images'}, inplace=True)

In [ ]:
all_proj_df_imagery = all_proj_df.merge(all_proj_df_imagery_usable_baseline_summary , on=['Project', 'poly_name'], how='left')
all_proj_df_imagery = all_proj_df_imagery.merge(all_proj_df_imagery_usable_ev_summary, on=['Project', 'poly_name'], how='left')

all_proj_df_imagery['available_baseline_images'] = all_proj_df_imagery['available_baseline_images'].replace({np.nan:int(0)})
all_proj_df_imagery['available_ev_images'] = all_proj_df_imagery['available_ev_images'].replace({np.nan:int(0)})

In [ ]:
baseline_available = all_proj_df_imagery[all_proj_df_imagery['available_baseline_images'] > 0].groupby('Project')['Area_ha'].sum().reset_index().rename(columns={'Area_ha': 'BaselineImageryCoverage_ha'})
ev_available = all_proj_df_imagery[all_proj_df_imagery['available_ev_images'] > 0].groupby('Project')['Area_ha'].sum().reset_index().rename(columns={'Area_ha': 'EVImageryCoverage_ha'})

In [ ]:
all_proj_md_imagery = practice_target_sys_by_project.merge(baseline_available, on='Project', how='left').merge(ev_available, on='Project', how='left')
all_proj_md_imagery['BaselineImageryCoverage_ha'] =all_proj_md_imagery['BaselineImageryCoverage_ha'].replace({np.nan:int(0)})
all_proj_md_imagery['EVImageryCoverage_ha'] =all_proj_md_imagery['EVImageryCoverage_ha'].replace({np.nan:int(0)})

In [ ]:
all_proj_md_imagery

# Landscape boundaries 

In [ ]:
landscape_data_path = "../data/landscape_boundaries/"
landscape_files = os.listdir(landscape_data_path)

In [ ]:
landscape_list = ['KRB.shp','GRV.shp', 'GhanaCocoaBelt.shp']
landscape_poly_list = []
for landscape in landscape_list:
    shapefile_features =[]
    with fiona.collection(f"{landscape_data_path}/{landscape}", driver='ESRI Shapefile') as shp_in:
        shp_crs = pyproj.crs.CRS(shp_in.crs)
        shpschema = shp_in.schema.copy()
        for obj in shp_in:
            shapefile_features.append(obj)
    shp_gdf = gpd.GeoDataFrame.from_features([feature for feature in shapefile_features], crs=shp_crs)
    shp_gdf['Landscape'] = landscape.split('.')[0]
    landscape_poly_list.append(shp_gdf)
all_landscape_df = gpd.GeoDataFrame(pd.concat(landscape_poly_list, ignore_index=True))

all_landscape_df_sub = all_landscape_df[all_landscape_df.index.isin([0,1,5])].reset_index()
all_landscape_df_sub['geometry'] =all_landscape_df_sub['geometry'].to_crs({'init': 'epsg:3857'})

In [ ]:
all_proj_short_df_copy= all_proj_df.__deepcopy__()
all_proj_short_df_copy['geometry'] = all_proj_short_df_copy['geometry'].to_crs({'init': 'epsg:3857'})
all_proj_short_df_copy['KRB_overlap_ha'] = all_proj_short_df_copy['geometry'].apply(lambda x: get_intersection_area(all_landscape_df_sub['geometry'][0], x))
all_proj_short_df_copy['GRV_overlap_ha'] = all_proj_short_df_copy['geometry'].apply(lambda x: get_intersection_area(all_landscape_df_sub['geometry'][1], x))
all_proj_short_df_copy['GhanaCocoaBelt_overlap_ha'] = all_proj_short_df_copy['geometry'].apply(lambda x: get_intersection_area(all_landscape_df_sub['geometry'][2], x))

In [ ]:
landscape_cols = ['KRB_overlap_ha', 'GRV_overlap_ha', 'GhanaCocoaBelt_overlap_ha']
landscape_by_project = all_proj_short_df_copy.groupby('Project')[landscape_cols].sum().reset_index()

In [ ]:
all_proj_md_imagery_landscape = all_proj_md_imagery.merge(landscape_by_project, on='Project', how='left')

In [ ]:
(all_proj_md_imagery_landscape.loc[:, all_proj_md_imagery_landscape.columns != 'geometry']).to_csv("../data/bucketing_output/bucketing_features_project.csv", index=False)